# Solution: Transforming Time-Series Data in pandas

You have a clean daily bike-share series. Your job is to compute rolling statistics, test for stationarity, and apply differencing to prepare the series for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [ ]:
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
# Clean: deduplicate, fix October 2023 unit error, remove negatives, fill gaps
df = df.groupby("date", as_index=False)["rides"].mean()
df["date"] = pd.to_datetime(df["date"])
mask = (df["date"].dt.year == 2023) & (df["date"].dt.month == 10) & (df["rides"] < 1000)
df.loc[mask, "rides"] = df.loc[mask, "rides"] * 24
df.loc[df["rides"] < 0, "rides"] = np.nan
df = df.set_index("date").asfreq("D")
df["rides"] = df["rides"].interpolate(method="time")
rides = df["rides"]
print(f"Clean series: {len(rides)} days")

## Task 1: Compute Rolling Statistics

Compute the 30-day rolling mean and rolling standard deviation.

In [ ]:
def compute_rolling_stats(series, window=30):
    return series.rolling(window).mean(), series.rolling(window).std()

rmean, rstd = compute_rolling_stats(rides)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color="lightgray", linewidth=0.5)
axes[0].plot(rmean.index, rmean.values, color="black")
axes[0].set_ylabel("Rides")
axes[0].set_title("30-day rolling mean")
axes[1].plot(rstd.index, rstd.values, color="tab:orange")
axes[1].set_ylabel("Rides")
axes[1].set_title("30-day rolling std")
plt.tight_layout()
plt.show()

## Task 2: Run the ADF Stationarity Test

Use `adfuller` with `autolag="AIC"`. Return a dict with `test_statistic`, `p_value`, `lags_used`, `critical_values`.

In [ ]:
def run_adf_test(series):
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "test_statistic": result[0],
        "p_value": result[1],
        "lags_used": result[2],
        "critical_values": result[4],
    }

adf = run_adf_test(rides)
print(f"ADF statistic: {adf['test_statistic']:.4f}")
print(f"p-value: {adf['p_value']:.6f}")
print(f"Stationary: {'yes' if adf['p_value'] < 0.05 else 'no'}")

## Task 3: Apply First Differencing

Use `.diff()` to remove the trend. Drop the resulting NaN.

In [ ]:
def apply_differencing(series):
    return series.diff().dropna()

diffed = apply_differencing(rides)

In [ ]:
adf_diff = run_adf_test(diffed)
print(f"Differenced ADF p-value: {adf_diff['p_value']:.6f}")
print(f"Stationary: {'yes' if adf_diff['p_value'] < 0.05 else 'no'}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color="black", linewidth=0.5)
axes[0].set_ylabel("Rides")
axes[0].set_title("Original series")
axes[1].plot(diffed.index, diffed.values, color="tab:green", linewidth=0.5)
axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_ylabel("Daily change")
axes[1].set_title("First-differenced series")
plt.tight_layout()
plt.show()

## Task 4: Train/Test Split

Split chronologically, holding out the last 90 days for testing.

In [ ]:
def train_test_split(series, test_days=90):
    return series.iloc[:-test_days], series.iloc[-test_days:]

train, test = train_test_split(rides)
print(f"Train: {len(train)} days")
print(f"Test: {len(test)} days")